# Connect a notebook to your project's kernel

*Python Foundations · Topic 10.2*

The marketing team is building a custom channel group in GA4. Its Organic Social rule needs the exact session source values of Facebook and Instagram, and everything else with the medium referral stays Referral. Build the lists from April 2024's export, with sets. This workbook needs `tldextract`, which is in your `referral-report` project.

1. Choose **Kernel → Restart Kernel and Run All Cells…**. Read the output of the first three code cells: which Python runs this notebook, what `%pip` says, and where it stops.
2. Delete the `%pip` cell. It's from Colab, and `tldextract` is already in `referral-report`.
3. Leave JupyterLab running. In a second terminal window, in `referral-report`, register the project's Python as a kernel called `referral-report`.
4. Choose **Kernel → Change Kernel…** and choose **referral-report**. Choose **Kernel → Restart Kernel and Run All Cells…** again. The first cell should now show a path in `referral-report`'s `.venv`, and the cell that reads the file should work.
5. In the first cell marked ✏️, create `referral_hostnames`: the set of `session_source` values of the rows whose `session_medium` is `referral`, with a set comprehension.
6. In the second cell marked ✏️, create `facebook_hosts` and `instagram_hosts`: the hostnames in `referral_hostnames` whose `site_name()` is `facebook` and `instagram`.
7. In the last cell marked ✏️, create `social_hosts`, the hostnames in either set, and `other_hosts`, the referral hostnames that aren't social. End the cell with `sorted(other_hosts)`.
8. Run the **Check your work** cell. Then choose **Kernel → Restart Kernel and Run All Cells…** to check that you get the same result, save the notebook and shut JupyterLab down with **File → Shut Down**.

**Expected output**: `referral_hostnames` has 11 hostnames, `facebook_hosts` has 4, `instagram_hosts` 2 and `social_hosts` 6, and the last cell shows the five `.example` blogs.

In [ ]:
import sys

sys.executable

In [ ]:
# ✏️ This cell is from Colab. Delete it.
%pip install tldextract

In [ ]:
import csv

import tldextract

CSV_PATH = "data/sessions_export_2024_04.csv"

with open(CSV_PATH, newline="", encoding="utf-8") as file:
    rows = list(csv.DictReader(file))


def site_name(hostname):
    return tldextract.extract(hostname).domain.lower()


len(rows)

In [ ]:
# ✏️ Create referral_hostnames with a set comprehension


In [ ]:
# ✏️ Create facebook_hosts and instagram_hosts


In [ ]:
# ✏️ Create social_hosts and other_hosts, and end the cell with sorted(other_hosts)


In [ ]:
# Check your work: run this cell last. If you change anything above, run the cells again from the top first.
# This cell only reads your variables. It doesn't change anything.
def check_your_work():
    import sys
    from pathlib import Path
    found = globals()
    results = []

    folder = Path.cwd()
    project = next((path for path in [folder, *folder.parents] if (path / "pyproject.toml").exists()), None)
    prefix = Path(sys.prefix).resolve()
    if project is None:
        results.append((False, f"This notebook isn't in a uv project's folder, but in {folder}. Move it into referral-report, with its data folder."))
    elif project.name != "referral-report":
        results.append((False, f"This notebook is in the {project.name} project. The exercise uses referral-report, from topic 9.3.: run get_files.py in that folder."))
    elif prefix == (project / ".venv").resolve():
        results.append((True, "The kernel is referral-report's .venv."))
    elif "archive-v0" in prefix.parts or prefix.parent.name == "tools":
        results.append((False, f"The kernel is {sys.executable}, the Python of the JupyterLab you started with uvx, not the project's. Register referral-report's Python as a kernel (step 3), then choose Kernel → Change Kernel… → referral-report."))
    else:
        results.append((False, f"The kernel is {sys.executable}, not referral-report's .venv. Choose Kernel → Change Kernel… → referral-report. If it isn't in the list, register it first (step 3)."))

    try:
        from jupyter_client.kernelspec import KernelSpecManager
        registered = KernelSpecManager().find_kernel_specs()
    except Exception:
        registered = None
    if registered is not None and "referral-report" not in registered:
        results.append((False, "There's no kernel called referral-report. In a terminal, in referral-report, run: uv run python -m ipykernel install --user --name referral-report --display-name referral-report"))
    elif registered is not None:
        results.append((True, "referral-report is registered as a kernel."))

    module = sys.modules.get("tldextract")
    if module is None:
        results.append((False, "tldextract isn't imported. Run the cell that reads the file, after you've changed the kernel."))
    elif project is not None and not Path(module.__file__).resolve().is_relative_to((project / ".venv").resolve()):
        results.append((False, f"tldextract imports from {Path(module.__file__).parent}, not from the project's .venv. Change the kernel to referral-report, then restart and run all."))
    else:
        results.append((True, "tldextract imports from the project's .venv."))

    hostnames = found.get("referral_hostnames")
    expected_hostnames = {"deals-daily.example", "facebook.com", "giftguide.example", "instagram.com", "l.facebook.com", "l.instagram.com", "lm.facebook.com", "m.facebook.com", "stylebloggers.example", "threadnotes.example", "wardrobewire.example"}
    if hostnames is None:
        results.append((False, "referral_hostnames isn't defined. Run the cell that creates it."))
    elif isinstance(hostnames, list):
        results.append((False, f"referral_hostnames is a list of {len(hostnames)} values. Use curly braces instead of square brackets, so it's a set and each hostname appears once."))
    elif not isinstance(hostnames, set):
        results.append((False, f"referral_hostnames should be a set, but it is a {type(hostnames).__name__}."))
    elif hostnames == expected_hostnames:
        results.append((True, "referral_hostnames has the 11 referral hostnames."))
    elif "facebook" in hostnames or "example" in hostnames:
        results.append((False, "referral_hostnames has site names. Keep the hostnames as they are in session_source: row[\"session_source\"]."))
    elif "google" in hostnames or "(direct)" in hostnames:
        results.append((False, f"referral_hostnames has {len(hostnames)} values, including sources that aren't referrals. Keep only the rows where row[\"session_medium\"] == \"referral\"."))
    else:
        results.append((False, f"referral_hostnames should be {sorted(expected_hostnames)}, but it is {sorted(hostnames, key=str)}."))

    facebook = {"facebook.com", "l.facebook.com", "lm.facebook.com", "m.facebook.com"}
    instagram = {"instagram.com", "l.instagram.com"}
    for name, expected, site in [("facebook_hosts", facebook, "facebook"), ("instagram_hosts", instagram, "instagram")]:
        value = found.get(name)
        if value is None:
            results.append((False, f"{name} isn't defined. Run the cell that creates it."))
        elif value == expected:
            results.append((True, f"{name} has the {len(expected)} {site.capitalize()} hostnames."))
        elif value == {site}:
            results.append((False, f"{name} is {{{site!r}}}, the site's name. Keep the hostnames whose site_name() is {site!r}: {{hostname for hostname in referral_hostnames if site_name(hostname) == {site!r}}}."))
        elif not isinstance(value, set):
            results.append((False, f"{name} should be a set, but it is a {type(value).__name__}."))
        else:
            results.append((False, f"{name} should be {sorted(expected)}, but it is {sorted(value, key=str)}."))

    social = found.get("social_hosts")
    if social is None:
        results.append((False, "social_hosts isn't defined. Run the cell that creates it."))
    elif social == facebook | instagram:
        results.append((True, "social_hosts has the 6 Facebook and Instagram hostnames."))
    elif social == set():
        results.append((False, "social_hosts is empty: no hostname is in both sets. Use |, the union, for the hostnames in either set: facebook_hosts | instagram_hosts."))
    else:
        results.append((False, f"social_hosts should be the 6 hostnames in facebook_hosts or instagram_hosts, but it is {sorted(social, key=str) if isinstance(social, set) else social!r}."))

    other = found.get("other_hosts")
    expected_other = expected_hostnames - facebook - instagram
    if other is None:
        results.append((False, "other_hosts isn't defined. Run the cell that creates it."))
    elif other == expected_other:
        results.append((True, "other_hosts has the 5 referral hostnames that aren't social."))
    elif other == set():
        results.append((False, "other_hosts is empty. The order matters in a difference: referral_hostnames - social_hosts."))
    else:
        results.append((False, f"other_hosts should be {sorted(expected_other)}, but it is {sorted(other, key=str) if isinstance(other, set) else other!r}."))

    for passed, message in results:
        print(("✅ " if passed else "❌ ") + message)
    if all(passed for passed, message in results):
        print("\nEverything matches. Now choose Kernel → Restart Kernel and Run All Cells…, and check that you get the same result.")
    else:
        print("\nFix the first ❌, run the cells above again from the top, then run this cell again.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

Delete the `%pip` cell.

```python
referral_hostnames = {row["session_source"] for row in rows if row["session_medium"] == "referral"}
len(referral_hostnames)
```

```python
facebook_hosts = {hostname for hostname in referral_hostnames if site_name(hostname) == "facebook"}
instagram_hosts = {hostname for hostname in referral_hostnames if site_name(hostname) == "instagram"}
```

```python
social_hosts = facebook_hosts | instagram_hosts
other_hosts = referral_hostnames - social_hosts
sorted(other_hosts)
```

```text
uv run https://raw.githubusercontent.com/team-simmer/python-foundations/main/lesson-10/get_files.py 10.2
uvx jupyter lab
```

```text
uv run python -m ipykernel install --user --name referral-report --display-name referral-report
```

</details>